# 그래프가 스스로 멈추기 · interrupt 와 Command

강의 오후 ④ 앞부분 (Academy 모듈 3 · L4 Dynamic breakpoints)

| 칸 | 보여 주는 것 |
| --- | --- |
| ① 스스로 멈춘다 | tools 노드 안에서, 사전에 없는 단어일 때만 `interrupt(값)` 으로 멈춘다. 사전에 있는 단어는 그냥 지나간다 |
| ② 답을 넣어 이어 간다 | `new Command({ resume: answer })` 으로 이어 가면, 그 답이 `interrupt()` 가 돌려주는 값이 된다 |
| ③ 노드가 처음부터 다시 돈다 | 이어 가면 멈췄던 노드가 처음부터 다시 돈다. `interrupt()` 앞의 코드는 두 번 돈다 |
| ④ 사람에게 직접 묻기 | 멈춘 이유를 보여 주고 `prompt()` 로 받은 답을 그대로 넣는다 |

오후 ③ 의 breakpoint 는 돌리기 전에 이름으로 정한 노드(compile 이든 실행 옵션이든)에 닿으면 **늘** 멈췄습니다. 이번에는 노드가 상황을 보고 **필요할 때만** 멈춥니다. 

In [ ]:
import "npm:@langchain/core@1.2.14";
import { Annotation, StateGraph, START, END, MemorySaver, interrupt, Command } from "npm:@langchain/langgraph@1.4.18";

// 접속 정보: 실습과 같은 lab/.env 를 읽는다 (OpenRouter · qwen/qwen3-32b). 키는 화면에 찍지 않는다
const ENV: Record<string, string> = {};
try {
  for (const line of Deno.readTextFileSync("../lab/.env").split("\n")) {
    const m = line.match(/^\s*([A-Z_]+)\s*=\s*(.*)\s*$/);
    if (m) ENV[m[1]] = m[2];
  }
} catch { /* lab/.env 가 없으면 아래 기본값 */ }
const BASE_URL = ENV.LLM_BASE_URL ?? "https://openrouter.ai/api/v1";
const API_KEY  = ENV.LLM_API_KEY  ?? "";
const MODEL    = ENV.LLM_MODEL    ?? "qwen/qwen3-32b";
const OPTIONS  = { temperature: 0 };   // 시연이 매번 같게
console.log({ BASE_URL, MODEL, 키: API_KEY ? "있음" : "없음 — lab/.env 에 LLM_API_KEY 를 넣는다" });

// LangSmith (강사 시연용): demo/.env 에 키가 있으면 실행 기록(trace)이 LangSmith 에 남는다. 실습은 lab/.env 를 써서 남지 않는다
import { traceable } from "npm:langsmith@0.11.1/traceable";
try {
  for (const line of Deno.readTextFileSync(".env").split("\n")) {
    const m = line.match(/^\s*(LANGSMITH_[A-Z_]+)\s*=\s*(?:"([^"]*)"|([^#\s]*))/);   // "따옴표 값" 은 그대로, 아니면 # 주석 앞까지
    const value = m?.[2] ?? m?.[3];
    if (m && value) Deno.env.set(m[1], value);
  }
} catch { /* demo/.env 가 없으면 기록하지 않는다 */ }
if (!Deno.env.get("LANGSMITH_API_KEY")) Deno.env.set("LANGSMITH_TRACING", "false");   // 키가 없으면 끈다
// 모델을 부르는 함수를 감싸면 trace 에 "모델 호출" 한 칸으로 남는다. 기록을 안 켜면 그냥 원래 함수로 돈다
const asLLM = <F extends (...a: any[]) => Promise<any>>(fn: F, name: string) => traceable(fn, {
  name, run_type: "llm",
  processInputs: (i: any) => ({ messages: i.args ? i.args[0] : i.input }),
  processOutputs: (o: any) => ({ choices: [{ message: typeof o.outputs === "string" ? { role: "assistant", content: o.outputs } : o }] }),
});
console.log({ LangSmith: Deno.env.get("LANGSMITH_TRACING") === "true" && Deno.env.get("LANGSMITH_API_KEY") ? `기록함 · project ${Deno.env.get("LANGSMITH_PROJECT") ?? "default"}` : "기록 안 함 (demo/.env)" });

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

## 준비: 튜터 Agent

모델 부르기, 사전 도구, `pretty` 는 앞 노트북들과 같습니다. 이번에는 tools 노드만 새로 씁니다.

In [ ]:
// 오전에 만든 chat() 그대로 — 다시 보낼 칸(role · content · tool_calls)만 돌려준다
async function callModel(messages: any[], tools?: any[]) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, tools, ...OPTIONS }),
  });
  const { role, content, tool_calls } = (await res.json()).choices[0].message;
  return { role, content: content?.trim() ?? null, tool_calls };
}
const chat = asLLM(callModel, "chat");   // LangSmith 에 모델 호출로 남긴다

// 도구 하나: 단어 뜻과 쉬운 말 찾기 (작은 사전이라 시연이 매번 같다)
const DICT: Record<string, { 뜻: string; 쉬운말: string }> = {
  ubiquitous: { 뜻: "어디에나 있는", 쉬운말: "everywhere" },
  mitigate:   { 뜻: "완화하다, 줄이다", 쉬운말: "reduce" },
  meticulous: { 뜻: "꼼꼼한", 쉬운말: "careful" },
  detrimental:{ 뜻: "해로운", 쉬운말: "harmful" },
};
const tools = [{
  type: "function",
  function: {
    name: "lookup_word",
    description: "영어 단어 하나의 뜻과, 바꿔 쓸 쉬운 말을 사전에서 찾는다",
    parameters: { type: "object", properties: { word: { type: "string", description: "찾을 영어 단어 하나" } }, required: ["word"] },
  },
}];
function lookupWord(word: string) {
  const w = word.toLowerCase().trim();
  return DICT[w] ? { word: w, ...DICT[w] } : { word: w, error: "사전에 없는 단어" };
}

const SYSTEM = { role: "system", content: "너는 IELTS 쓰기를 돕는 영어 튜터다. 단어 뜻은 추측하지 말고 lookup_word 로 확인한다. 한국어로 짧게 답한다." };
const ask = (q: string) => ({ messages: [SYSTEM, { role: "user", content: q }] });
// 메시지를 보기 좋게 찍는다 (tutor-stream 노트북의 pretty 그대로)
const TITLE: Record<string, string> = { system: "System Message", user: "Human Message", assistant: "Ai Message", tool: "Tool Message" };
function pretty(m: any) {
  const title = ` ${TITLE[m.role]} `;
  const left = Math.floor((60 - title.length) / 2);
  console.log("=".repeat(left) + title + "=".repeat(60 - title.length - left));
  if (m.tool_calls?.length) {
    const calls = m.tool_calls.map((c: any) => `${c.function.name}(${c.function.arguments})`);
    console.log("Tool Calls:", calls.join(", "));
  } else console.log(m.content);
}

In [ ]:
// 오후 ① 의 튜터 Agent: tutor ⇢ tools → tutor
const State = Annotation.Root({
  messages: Annotation<any[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
});
async function tutor(state: typeof State.State) {
  return { messages: [await chat(state.messages, tools)] };
}
function route(state: typeof State.State) {
  return state.messages.at(-1).tool_calls?.length ? "tools" : END;
}

// stream 으로 돌리며 State 를 받을 때마다 찍는다. 멈추면 interrupt() 에 넘긴 값을 보여 준다
async function run(graph: any, input: any, thread: any) {
  for await (const state of await graph.stream(input, { ...thread, streamMode: "values" })) {
    if (state.__interrupt__) console.log("⏸ __interrupt__", JSON.stringify(state.__interrupt__.map((i: any) => i.value)));
    else {
      for (const m of state.messages) if (m.role !== "system") pretty(m);   // system 메시지는 매번 같으니 뺀다
      console.log("-".repeat(60));   // State 하나 끝
    }
  }
}

## ① 그래프가 스스로 멈춘다 — interrupt()

tools 노드 안에서 사전에 없는 단어를 만나면 `interrupt(값)` 을 부릅니다. 그 자리에서 그래프가 멈추고, 넘긴 값이 `__interrupt__` 에 실려 밖으로 나갑니다. 사전에 있는 단어면 `interrupt()` 를 부르지 않으니 멈추지 않습니다.

compile 에는 `interruptBefore` 를 적지 않습니다. 멈춘 자리를 저장해야 하니 checkpointer 는 있어야 합니다.

In [ ]:
// ① tools 노드가 스스로 멈춘다: 사전에 없는 단어면 interrupt() 로 사람에게 묻는다
function toolsNode(state: typeof State.State) {
  console.log("▶ tools 시작");   // 이어 가면 노드가 처음부터 다시 돈다. 그때 이 줄이 다시 찍힌다
  const calls = state.messages.at(-1).tool_calls;
  return { messages: calls.map((c: any) => {
    const word = JSON.parse(c.function.arguments).word.toLowerCase().trim();
    console.log(`  사전 찾기: ${word}`);
    let result: any = lookupWord(word);
    if (result.error) {
      // 여기서 멈춘다. 이어 갈 때는 사람이 넣은 답이 돌아온다
      const meaning = interrupt({ question: "사전에 없는 단어입니다. 뜻을 알려 주세요", word });
      result = { word, meaning, source: "사람" };
    }
    return { role: "tool", tool_call_id: c.id, content: JSON.stringify(result) };
  }) };
}
const agent = new StateGraph(State)
  .addNode("tutor", tutor)
  .addNode("tools", toolsNode)
  .addEdge(START, "tutor")
  .addConditionalEdges("tutor", route, ["tools", END])
  .addEdge("tools", "tutor")
  .compile({ checkpointer: new MemorySaver() });   // interruptBefore 는 없다
await showGraph(agent);

// 사전에 있는 단어: 멈추지 않는다
const thread1 = { configurable: { thread_id: "interrupt-1" } };
await run(agent, ask("ubiquitous 뜻을 사전에서 찾아 줘"), thread1);

**돌려 보면** 사전에 있는 단어라 `interrupt()` 를 부르지 않았습니다. 멈추지 않고 답까지 갑니다.

이번에는 사전에 없는 단어 serendipity 를 물어보겠습니다.

In [ ]:
// 사전에 없는 단어: tools 노드 안에서 멈춘다
const thread2 = { configurable: { thread_id: "interrupt-2" } };
await run(agent, ask("serendipity 뜻을 사전에서 찾아 줘"), thread2);
console.log("다음 노드:", (await agent.getState(thread2)).next);

**돌려 보면** 도구 요청 State 다음에 `__interrupt__` 가 오고 멈춥니다. 오후 ③ 에서는 `__interrupt__` 가 빈 배열이었는데, 이번에는 `interrupt()` 에 넘긴 값 `{ question, word }` 가 들어 있습니다. 왜 멈췄는지를 그래프가 직접 알려 줍니다.

`next` 는 `["tools"]` 입니다. tools 노드가 끝나지 못하고 멈춰 있다는 뜻입니다.

## ② 답을 넣어 이어 간다 — Command

이어 갈 때는 null 대신 `new Command({ resume: answer })` 을 넣습니다. 그러면 멈췄던 `interrupt()` 가 그 답을 돌려줍니다. 오후 ③ 의 human_feedback 노드 + `updateState(…, asNode)` 가 하던 일을 노드 안 한 줄이 합니다.

In [ ]:
// ② 이어 가기: 사람의 답을 Command 로 넣는다. interrupt() 가 이 값을 돌려준다
await run(agent, new Command({ resume: "뜻밖의 행운, 우연히 좋은 것을 발견함" }), thread2);

**돌려 보면** 맨 위에 `▶ tools 시작` 이 한 번 더 찍힙니다. 이어 갈 때 tools 노드가 **처음부터 다시** 돌았기 때문입니다. 이번에는 `interrupt()` 가 멈추지 않고 사람의 답을 돌려줘서, 도구 결과에 `source: "사람"` 이 붙었습니다. tutor 는 그 뜻으로 답합니다.

## ③ 노드가 처음부터 다시 돈다

단어 두 개를 한 번에 찾게 하면 차이가 잘 보입니다. 하나는 사전에 있고, 하나는 없습니다.

In [ ]:
// ③ 사전에 있는 단어와 없는 단어를 같이: 이어 가면 앞의 단어를 다시 찾는다
const thread3 = { configurable: { thread_id: "interrupt-3" } };
await run(agent, ask("ubiquitous 랑 serendipity 뜻을 사전에서 찾아 줘"), thread3);

In [ ]:
await run(agent, new Command({ resume: "뜻밖의 행운" }), thread3);

**돌려 보면** 이어 갈 때 `사전 찾기: ubiquitous` 가 또 찍힙니다. 멈추기 전에 이미 찾은 단어인데, 노드가 처음부터 다시 돌면서 한 번 더 찾았습니다.

`interrupt()` 는 await 로 기다리는 게 아니라 **예외를 던져** 노드를 그 자리에서 끝냅니다. 그래서 이어 갈 때는 노드를 처음부터 다시 돌리고, 이번에는 `interrupt()` 가 답을 돌려줍니다. `interrupt()` 앞에 있는 코드는 이어 갈 때 다시 돕니다. 사전 찾기는 두 번 해도 괜찮지만, 청구 · 메일 보내기처럼 되돌리기 어려운 일을 `interrupt()` 앞에 두면 두 번 일어납니다. 그런 일은 `interrupt()` **뒤에** 둡니다. 그리고 `interrupt()` 를 try/catch 로 감싸 예외를 삼키면 멈추지 않습니다.

## ④ 사람에게 직접 묻기

멈춘 이유는 `getState` 의 `tasks` 에도 들어 있습니다. 그 질문을 보여 주고, `prompt()` 로 받은 답을 그대로 `Command` 에 넣습니다.

In [ ]:
// ④ 멈추면 interrupt 에 실린 질문을 보여 주고, 입력을 Command 로 넣는다
const thread4 = { configurable: { thread_id: "interrupt-4" } };
await run(agent, ask("serendipity 뜻을 사전에서 찾아 줘"), thread4);
const [pending] = (await agent.getState(thread4)).tasks.flatMap((t: any) => t.interrupts);
const answer = prompt(`${pending.value.question} (${pending.value.word})`) ?? "모름";   // 노트북 위쪽에 입력창이 뜬다. 그냥 닫으면 "모름"
await run(agent, new Command({ resume: answer }), thread4);

**돌려 보면** 입력창에 적은 뜻으로 tutor 가 답합니다.

정리하면 두 가지가 바뀌었습니다. 멈출지 말지를 **노드가 상황을 보고** 정하고, 사람의 답은 `Command({ resume })` 으로 **노드 안으로** 바로 들어갑니다.